# (Homework) Week 6 - DataScience Bootcamp Fall 2025

All solution cells are replaced with `# TODO` placeholders so you can fill them in.

**Name:** \
**Email:**

---

### Problem 1: Dataset Splitting

1. You have recordings of 44 phones from 100 people; each person records ~200 phones/day for 5 days.
   - Design a valid training/validation/test split strategy that ensures the model generalizes to **new speakers**.

2. You now receive an additional dataset of 10,000 phone recordings from **Kilian**, a single speaker.
   - You must train a model that performs well **specifically for Kilian**, while also maintaining generalization.

*Describe your proposed split strategy and reasoning.* (Theory)

In [1]:
Proposed Split Strategy: Speaker-Based Split
Training Set:   70 speakers (70% of people)
Validation Set: 15 speakers (15% of people)
Test Set:       15 speakers (15% of people)
Key principle: Split by speakers, not by recordings
Reasoning:

No speaker overlap between train/val/test ensures we test generalization to new people
If we randomly split recordings, the same speaker could appear in all sets, leading to data leakage
The model would learn speaker-specific patterns rather than general phone recognition
This split simulates real-world deployment where the model encounters new speakers

Implementation details:

Randomly assign each of the 100 speakers to train/val/test
All ~1000 recordings from each speaker stay together in their assigned set
Stratify by relevant factors if available (gender, accent, age) to ensure balanced representation

### Problem 2: K-Nearest Neighbors

1. **1-NN Classification:** Given dataset:

   Positive: (1,2), (1,4), (5,4)

   Negative: (3,1), (3,2)

   Plot the 1-NN decision boundary and classify new points visually.

2. **Feature Scaling:** Consider dataset:

   Positive: (100,2), (100,4), (500,4)

   Negative: (300,1), (300,2)

   What would the 1-NN classify point (500,1) as **before and after scaling** to [0,1] per feature?

3. **Handling Missing Values:** How can you modify K-NN to handle missing features in a test point?

4. **High-dimensional Data:** Why can K-NN still work well for images even with thousands of pixels?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi, voronoi_plot_2d
from sklearn.preprocessing import MinMaxScaler

# Dataset
positive = np.array([[1,2], [1,4], [5,4]])
negative = np.array([[3,1], [3,2]])

# Combine all points
all_points = np.vstack([positive, negative])
labels = ['Positive']*3 + ['Negative']*2

# Create figure
plt.figure(figsize=(10, 8))

# Plot training points
plt.scatter(positive[:, 0], positive[:, 1], c='blue', s=200, marker='o',
            edgecolors='black', linewidths=2, label='Positive', zorder=3)
plt.scatter(negative[:, 0], negative[:, 1], c='red', s=200, marker='s',
            edgecolors='black', linewidths=2, label='Negative', zorder=3)

# Create a mesh for decision boundary
x_min, x_max = -0.5, 6.5
y_min, y_max = 0, 5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500),
                      np.linspace(y_min, y_max, 500))

# Classify each point in the mesh using 1-NN
def classify_1nn(point, train_points, train_labels):
    distances = np.sqrt(np.sum((train_points - point)**2, axis=1))
    nearest_idx = np.argmin(distances)
    return train_labels[nearest_idx]

# Create labels array (0 for positive, 1 for negative)
train_labels = np.array([0, 0, 0, 1, 1])

# Classify each point in mesh
Z = np.zeros(xx.shape)
for i in range(xx.shape[0]):
    for j in range(xx.shape[1]):
        point = np.array([xx[i, j], yy[i, j]])
        Z[i, j] = classify_1nn(point, all_points, train_labels)

# Plot decision regions
plt.contourf(xx, yy, Z, alpha=0.3, levels=1, colors=['blue', 'red'])
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')

# Test some new points
test_points = np.array([[2, 3], [4, 2], [2, 1], [4, 4]])
for point in test_points:
    prediction = classify_1nn(point, all_points, train_labels)
    color = 'blue' if prediction == 0 else 'red'
    marker = 'o' if prediction == 0 else 's'
    plt.scatter(point[0], point[1], c=color, s=150, marker=marker,
                edgecolors='green', linewidths=3, alpha=0.7, label='Test point' if point[0] == 2 and point[1] == 3 else '')

plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('1-NN Decision Boundary', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.tight_layout()
plt.show()

# Print classifications for test points
print("Test Point Classifications:")
for point in test_points:
    prediction = classify_1nn(point, all_points, train_labels)
    label = "Positive" if prediction == 0 else "Negative"
    # Find nearest neighbor
    distances = np.sqrt(np.sum((all_points - point)**2, axis=1))
    nearest_idx = np.argmin(distances)
    nearest_point = all_points[nearest_idx]
    distance = distances[nearest_idx]
    print(f"  {point} → {label} (nearest: {nearest_point}, distance: {distance:.2f})")








############### Part 2 ##################

# Original dataset
positive_orig = np.array([[100,2], [100,4], [500,4]])
negative_orig = np.array([[300,1], [300,2]])
all_points_orig = np.vstack([positive_orig, negative_orig])
labels = np.array([0, 0, 0, 1, 1])  # 0=Positive, 1=Negative

# Test point
test_point_orig = np.array([[500, 1]])

print("="*60)
print("BEFORE SCALING")
print("="*60)

# Calculate distances to all points
distances_before = []
for i, point in enumerate(all_points_orig):
    dist = np.sqrt(np.sum((point - test_point_orig)**2))
    label = "Positive" if labels[i] == 0 else "Negative"
    distances_before.append((dist, label, point))
    print(f"Distance to {point} ({label}): {dist:.2f}")

distances_before.sort()
nearest_before = distances_before[0]
print(f"\n✓ Nearest neighbor: {nearest_before[2]} ({nearest_before[1]})")
print(f"  Distance: {nearest_before[0]:.2f}")
print(f"  Classification: {nearest_before[1]}")

print("\n" + "="*60)
print("AFTER SCALING [0,1]")
print("="*60)

# Scale features to [0,1]
scaler = MinMaxScaler()
all_points_scaled = scaler.fit_transform(all_points_orig)
test_point_scaled = scaler.transform(test_point_orig)

print(f"\nOriginal test point: {test_point_orig[0]}")
print(f"Scaled test point: {test_point_scaled[0]}")

print("\nScaled training points:")
for i, point in enumerate(all_points_scaled):
    label = "Positive" if labels[i] == 0 else "Negative"
    print(f"  {all_points_orig[i]} → {point} ({label})")

# Calculate distances after scaling
distances_after = []
for i, point in enumerate(all_points_scaled):
    dist = np.sqrt(np.sum((point - test_point_scaled)**2))
    label = "Positive" if labels[i] == 0 else "Negative"
    distances_after.append((dist, label, all_points_orig[i], point))
    print(f"Distance to {point} ({label}): {dist:.4f}")

distances_after.sort()
nearest_after = distances_after[0]
print(f"\n✓ Nearest neighbor: {nearest_after[2]} ({nearest_after[1]})")
print(f"  Distance: {nearest_after[0]:.4f}")
print(f"  Classification: {nearest_after[1]}")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Before scaling: Classified as {nearest_before[1]}")
print(f"After scaling:  Classified as {nearest_after[1]}")
print(f"\n{'⚠️ CLASSIFICATION CHANGED!' if nearest_before[1] != nearest_after[1] else '✓ Same classification'}")

### Problem 3: Part 1

You are given a fully trained Perceptron model with weight vector **w**, along with training set **D_TR** and test set **D_TE**.

1. Your co-worker suggests evaluating $h(x) = sign(w \cdot x)$ for every $(x, y)$ in D_TR and D_TE. Does this help determine whether test error is higher than training error?
2. Why is there no need to compute training error explicitly for the Perceptron algorithm?

In [ ]:
import numpy as np

# Assume we have trained perceptron with weight w
w = np.array([0.5, -0.3, 0.8])  # example weights

# Training set
X_train = np.array([[1, 2, 1], [2, 3, 1], ...])
y_train = np.array([1, -1, ...])

# Test set
X_test = np.array([[1.5, 2.5, 1], ...])
y_test = np.array([1, ...])

# Compute predictions
def predict(X, w):
    return np.sign(np.dot(X, w))

# Compute errors
train_predictions = predict(X_train, w)
test_predictions = predict(X_test, w)

train_error = np.mean(train_predictions != y_train)
test_error = np.mean(test_predictions != y_test)

print(f"Training Error: {train_error:.3f}")
print(f"Test Error: {test_error:.3f}")

if test_error > train_error:
    print("Test error is higher (overfitting)")
elif test_error < train_error:
    print("Test error is lower (unusual, possibly lucky split)")
else:
    print("Errors are equal")




### Problem 3: Two-point 2D Dataset (Part 2)

Run the Perceptron algorithm **by hand or in code** on the following data:

1. Positive class: (10, -2)
2. Negative class: (12, 2)

Start with $w_0 = (0, 0)$ and a learning rate of 1.

- Compute how many updates are required until convergence.
- Write down the sequence of $w_i$ vectors.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

# Data
X = np.array([[10, -2],   # Positive
              [12, 2]])    # Negative
y = np.array([1, -1])

# Perceptron algorithm with detailed logging
def perceptron_detailed(X, y, learning_rate=1.0, max_epochs=100):
    w = np.zeros(X.shape[1])
    w_history = [w.copy()]
    update_count = 0

    print("="*70)
    print("PERCEPTRON ALGORITHM - DETAILED EXECUTION")
    print("="*70)
    print(f"Initial weight: w₀ = {w}")
    print(f"Learning rate: η = {learning_rate}\n")

    for epoch in range(max_epochs):
        print(f"{'='*70}")
        print(f"ITERATION {epoch + 1}")
        print(f"{'='*70}")
        errors_in_epoch = 0

        for i in range(len(X)):
            # Compute prediction
            activation = np.dot(w, X[i])
            prediction = np.sign(activation) if activation != 0 else 0

            print(f"\nPoint {i+1}: x = {X[i]}, y = {y[i]:+d}")
            print(f"  Current weight: w = {w}")
            print(f"  Activation: w·x = {activation:.1f}")
            print(f"  Prediction: ŷ = {prediction:+.0f}")

            # Check if misclassified
            if prediction != y[i]:
                print(f"  ❌ MISCLASSIFIED (ŷ={prediction:+.0f} ≠ y={y[i]:+d})")

                # Update weights
                w_old = w.copy()
                w = w + learning_rate * y[i] * X[i]
                update_count += 1
                errors_in_epoch += 1

                print(f"  UPDATE #{update_count}:")
                print(f"    w_new = w_old + η·y·x")
                print(f"    w_new = {w_old} + {learning_rate}·{y[i]:+d}·{X[i]}")
                print(f"    w_new = {w_old} + {learning_rate * y[i] * X[i]}")
                print(f"    w_new = {w}")

                w_history.append(w.copy())
            else:
                print(f"  ✓ CORRECT (ŷ={prediction:+.0f} = y={y[i]:+d})")

        print(f"\nErrors in iteration {epoch + 1}: {errors_in_epoch}")

        # Check convergence
        if errors_in_epoch == 0:
            print(f"\n{'='*70}")
            print(f"✓ CONVERGED after {epoch + 1} iterations!")
            print(f"Total updates: {update_count}")
            print(f"Final weight: w = {w}")
            print(f"{'='*70}")
            break

    return w, w_history, update_count

# Run the algorithm
final_w, w_history, total_updates = perceptron_detailed(X, y)

# Print sequence of weights
print("\n" + "="*70)
print("SEQUENCE OF WEIGHT VECTORS")
print("="*70)
for i, w in enumerate(w_history):
    print(f"w_{i} = {w}")

print(f"\n📊 SUMMARY:")
print(f"   Total updates required: {total_updates}")
print(f"   Total iterations: {len(w_history) - 1}")
print(f"   Final weight vector: {final_w}")

# Visualization
plt.figure(figsize=(12, 5))

# Plot 1: Data points and decision boundaries
plt.subplot(1, 2, 1)
plt.scatter(X[0, 0], X[0, 1], c='blue', s=200, marker='o',
            edgecolors='black', linewidths=2, label='Positive (+1)', zorder=3)
plt.scatter(X[1, 0], X[1, 1], c='red', s=200, marker='s',
            edgecolors='black', linewidths=2, label='Negative (-1)', zorder=3)

# Plot decision boundaries for key weight vectors
x_line = np.linspace(8, 14, 100)
colors = ['gray', 'orange', 'purple', 'green']
labels_shown = set()

for i, w in enumerate([w_history[0], w_history[2], w_history[5], w_history[-1]]):
    if w[1] != 0:
        y_line = -(w[0] * x_line) / w[1]
        label = f'w_{i}' if i < len(w_history) - 1 else f'w_{i} (final)'
        if i == 0:
            label = 'w₀ (initial)'
        plt.plot(x_line, y_line, '--', alpha=0.6, linewidth=2,
                color=colors[min(i, len(colors)-1)], label=label)

plt.xlabel('x₁', fontsize=12)
plt.ylabel('x₂', fontsize=12)
plt.title('Perceptron Decision Boundaries', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(8, 14)
plt.ylim(-4, 4)

# Plot 2: Weight evolution
plt.subplot(1, 2, 2)
w_array = np.array(w_history)
iterations = range(len(w_history))
plt.plot(iterations, w_array[:, 0], 'o-', label='w₁ (first component)', linewidth=2, markersize=8)
plt.plot(iterations, w_array[:, 1], 's-', label='w₂ (second component)', linewidth=2, markersize=8)
plt.xlabel('Update Number', fontsize=12)
plt.ylabel('Weight Value', fontsize=12)
plt.title('Weight Components Over Time', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()




w = (2, -18)

### Problem 4: Reconstructing the Weight Vector

Given the log of Perceptron updates:

| x | y | count |
|---|---|--------|
| (0, 0, 0, 0, 4) | +1 | 2 |
| (0, 0, 6, 5, 0) | +1 | 1 |
| (3, 0, 0, 0, 0) | -1 | 1 |
| (0, 9, 3, 6, 0) | -1 | 1 |
| (0, 1, 0, 2, 5) | -1 | 1 |

Assume learning rate = 1 and initial weight $w_0 = (0, 0, 0, 0, 0)$.

Compute the final weight vector after all updates.

In [ ]:
#Todo

### Problem 5: Visualizing Perceptron Convergence

Implement a Perceptron on a small 2D dataset with positive and negative examples.

- Plot the data points.
- After each update, visualize the decision boundary.
- Show how it converges to a stable separator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Set random seed for reproducibility
np.random.seed(42)

# Generate a linearly separable 2D dataset
def generate_data(n_positive=15, n_negative=15):
    """Generate linearly separable 2D data"""
    # Positive class: centered around (3, 3)
    positive = np.random.randn(n_positive, 2) * 0.8 + [3, 3]

    # Negative class: centered around (1, 1)
    negative = np.random.randn(n_negative, 2) * 0.8 + [1, 1]

    # Combine data
    X = np.vstack([positive, negative])
    y = np.array([1] * n_positive + [-1] * n_negative)

    # Shuffle
    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]

    return X, y

# Perceptron class with visualization
class PerceptronVisualizer:
    def __init__(self, learning_rate=1.0):
        self.learning_rate = learning_rate
        self.w = None
        self.history = []  # Store (weights, misclassified_point, update_occurred)

    def fit(self, X, y, max_epochs=100):
        """Train perceptron and record history"""
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)

        # Record initial state
        self.history.append({
            'weights': self.w.copy(),
            'misclassified': None,
            'update': False,
            'epoch': 0,
            'point_idx': None
        })

        update_count = 0

        for epoch in range(max_epochs):
            errors = 0

            for i in range(n_samples):
                # Compute prediction
                activation = np.dot(self.w, X[i])
                prediction = 1 if activation >= 0 else -1

                # Check if misclassified
                if prediction != y[i]:
                    # Record before update
                    self.history.append({
                        'weights': self.w.copy(),
                        'misclassified': (X[i], y[i]),
                        'update': True,
                        'epoch': epoch + 1,
                        'point_idx': i,
                        'prediction': prediction
                    })

                    # Update weights
                    self.w += self.learning_rate * y[i] * X[i]
                    update_count += 1
                    errors += 1

                    # Record after update
                    self.history.append({
                        'weights': self.w.copy(),
                        'misclassified': (X[i], y[i]),
                        'update': False,
                        'epoch': epoch + 1,
                        'point_idx': i,
                        'update_num': update_count
                    })

            # If no errors, converged
            if errors == 0:
                print(f"✓ Converged after {epoch + 1} epochs and {update_count} updates!")
                break

        return self

    def predict(self, X):
        """Make predictions"""
        activation = np.dot(X, self.w)
        return np.where(activation >= 0, 1, -1)

    def plot_decision_boundary(self, ax, X, y, step_info, xlim, ylim):
        """Plot data points and decision boundary"""
        ax.clear()

        # Plot positive examples
        pos_mask = y == 1
        ax.scatter(X[pos_mask, 0], X[pos_mask, 1],
                  c='blue', s=100, marker='o',
                  edgecolors='black', linewidths=1.5,
                  label='Positive (+1)', zorder=3)

        # Plot negative examples
        neg_mask = y == -1
        ax.scatter(X[neg_mask, 0], X[neg_mask, 1],
                  c='red', s=100, marker='s',
                  edgecolors='black', linewidths=1.5,
                  label='Negative (-1)', zorder=3)

        # Highlight misclassified point if any
        if step_info['misclassified'] is not None:
            point, label = step_info['misclassified']
            color = 'yellow' if step_info['update'] else 'lightgreen'
            ax.scatter(point[0], point[1],
                      s=400, facecolors='none',
                      edgecolors=color, linewidths=4,
                      zorder=4)

            if step_info['update']:
                ax.text(point[0], point[1] + 0.3, '❌',
                       fontsize=20, ha='center', zorder=5)

        # Plot decision boundary
        w = step_info['weights']
        if w[1] != 0:  # Avoid division by zero
            x_line = np.array([xlim[0], xlim[1]])
            y_line = -(w[0] * x_line) / w[1]
            ax.plot(x_line, y_line, 'g-', linewidth=2.5,
                   label='Decision Boundary', zorder=2)

            # Plot margin lines (perpendicular to decision boundary)
            # Direction perpendicular to boundary
            normal = w / np.linalg.norm(w)

            # Shade regions
            xx, yy = np.meshgrid(np.linspace(xlim[0], xlim[1], 200),
                                np.linspace(ylim[0], ylim[1], 200))
            Z = w[0] * xx + w[1] * yy
            ax.contourf(xx, yy, Z, levels=[-1000, 0, 1000],
                       colors=['#ffcccc', '#ccccff'], alpha=0.3, zorder=1)

        # Set labels and title
        ax.set_xlabel('Feature 1 (x₁)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Feature 2 (x₂)', fontsize=12, fontweight='bold')

        # Create title with update info
        if 'update_num' in step_info:
            title = f"After Update #{step_info['update_num']}\n"
            title += f"Epoch {step_info['epoch']}, w = [{w[0]:.2f}, {w[1]:.2f}]"
        elif step_info['update']:
            pred = step_info.get('prediction', '?')
            title = f"Epoch {step_info['epoch']} - Misclassification Detected\n"
            title += f"Predicted: {pred:+d}, Actual: {step_info['misclassified'][1]:+d}"
        else:
            title = f"Initial State\nw = [{w[0]:.2f}, {w[1]:.2f}]"

        ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
        ax.legend(loc='upper right', fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_aspect('equal')

# Generate data
X, y = generate_data(n_positive=15, n_negative=15)

print("Dataset generated:")
print(f"  Total samples: {len(X)}")
print(f"  Positive examples: {np.sum(y == 1)}")
print(f"  Negative examples: {np.sum(y == -1)}")

# Train perceptron
perceptron = PerceptronVisualizer(learning_rate=1.0)
perceptron.fit(X, y)

# Get axis limits
xlim = [X[:, 0].min() - 0.5, X[:, 0].max() + 0.5]
ylim = [X[:, 1].min() - 0.5, X[:, 1].max() + 0.5]

# Create static plots showing key steps
n_steps_to_show = min(10, len(perceptron.history))
step_indices = np.linspace(0, len(perceptron.history) - 1, n_steps_to_show, dtype=int)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, step_idx in enumerate(step_indices):
    step_info = perceptron.history[step_idx]
    perceptron.plot_decision_boundary(axes[idx], X, y, step_info, xlim, ylim)

plt.tight_layout()
plt.savefig('perceptron_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Training completed with {len(perceptron.history)} recorded states")

# Plot final result with larger view
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
final_step = perceptron.history[-1]
perceptron.plot_decision_boundary(ax, X, y, final_step, xlim, ylim)
ax.set_title('Final Converged Decision Boundary', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('perceptron_final.png', dpi=150, bbox_inches='tight')
plt.show()

# Print weight evolution
print("\n" + "="*70)
print("WEIGHT VECTOR EVOLUTION")
print("="*70)
update_num = 0
for i, step in enumerate(perceptron.history):
    if i == 0:
        print(f"w₀ (initial): {step['weights']}")
    elif 'update_num' in step:
        update_num = step['update_num']
        print(f"w_{update_num} (after update {update_num}): {step['weights']}")

# Detailed analysis
print("\n" + "="*70)
print("CONVERGENCE ANALYSIS")
print("="*70)
final_weights = perceptron.history[-1]['weights']
print(f"Final weights: w = {final_weights}")
print(f"Decision boundary equation: {final_weights[0]:.2f}·x₁ + {final_weights[1]:.2f}·x₂ = 0")

# Verify all points are correctly classified
predictions = perceptron.predict(X)
accuracy = np.mean(predictions == y)
print(f"Final accuracy: {accuracy * 100:.1f}%")
print(f"Correctly classified: {np.sum(predictions == y)}/{len(y)}")

# Test on some new points
print("\n" + "="*70)
print("TESTING ON NEW POINTS")
print("="*70)
test_points = np.array([
    [2.5, 2.5],
    [1.0, 1.5],
    [3.5, 3.5],
    [0.5, 0.5]
])

for point in test_points:
    pred = perceptron.predict(point.reshape(1, -1))[0]
    activation = np.dot(perceptron.w, point)
    print(f"Point {point}: prediction = {pred:+d}, activation = {activation:.2f}")

# Create animation (optional - will create an animated GIF)
print("\n" + "="*70)
print("CREATING ANIMATION")
print("="*70)

fig, ax = plt.subplots(figsize=(10, 8))

def animate(frame):
    step_info = perceptron.history[frame]
    perceptron.plot_decision_boundary(ax, X, y, step_info, xlim, ylim)
    return ax,

# Create animation
anim = FuncAnimation(fig, animate, frames=len(perceptron.history),
                    interval=500, blit=False, repeat=True)

# Save as GIF (requires pillow)
try:
    anim.save('perceptron_learning.gif', writer='pillow', fps=2)
    print("✓ Animation saved as 'perceptron_learning.gif'")
except:
    print("⚠ Could not save animation (pillow may not be installed)")

plt.close()

# Summary statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)
total_updates = sum(1 for step in perceptron.history if step.get('update_num') is not None)
print(f"Total weight updates: {total_updates}")
print(f"Total states recorded: {len(perceptron.history)}")
print(f"Convergence achieved: {'Yes ✓' if accuracy == 1.0 else 'No ✗'}")

fig = plt.figure(figsize=(18, 12))

# Show first 12 significant steps
significant_steps = []
for i, step in enumerate(perceptron.history):
    if i == 0:  # Initial
        significant_steps.append(i)
    elif step.get('update_num') is not None and step['update_num'] <= 6:
        significant_steps.append(i)
    elif i == len(perceptron.history) - 1:  # Final
        significant_steps.append(i)

n_plots = min(12, len(significant_steps))
rows = 3
cols = 4

for plot_idx in range(n_plots):
    if plot_idx < len(significant_steps):
        ax = plt.subplot(rows, cols, plot_idx + 1)
        step_idx = significant_steps[plot_idx]
        step_info = perceptron.history[step_idx]
        perceptron.plot_decision_boundary(ax, X, y, step_info, xlim, ylim)

plt.suptitle('Perceptron Learning Process: Step-by-Step Convergence',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('perceptron_detailed_steps.png', dpi=150, bbox_inches='tight')
plt.show()